# Tech Challenge Fase 4 - Reprodução da frente de áudio

**Equipe:** Benicio Cardoso, Marcelo Viana, Nirton Afonso e Valkiria Faria.

Este notebook reproduz 1.180 áudios sintéticos com Whisper tiny, librosa e Random Forest. A avaliação nova separa famílias de frases e suas variações em treino, validação, teste e holdout cego. O modelo congelado aprende **somente o treino**. Resultados emocionais não são diagnósticos.

**Custo:** nenhum serviço pago. Uma GPU comum é suficiente; não é necessário selecionar A100. Execute em ordem. Salve ao Drive antes de encerrar o ambiente.


In [ ]:
from pathlib import Path
import os, subprocess, sys
EM_COLAB = 'google.colab' in sys.modules or Path('/content').exists()
if EM_COLAB:
    RAIZ = Path('/content/TechChallengeFase4')
    if not (RAIZ / '.git').exists():
        subprocess.run(['git', 'clone', '--branch', 'Nirton', '--single-branch', 'https://github.com/Val-Faria/TechChallengeFase4.git', str(RAIZ)], check=True)
else:
    RAIZ = Path.cwd()
    while not (RAIZ / 'scripts/audio_reproduce.py').exists() and RAIZ != RAIZ.parent:
        RAIZ = RAIZ.parent
os.chdir(RAIZ)
print('Projeto:', RAIZ)


In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', 'audio/requirements.txt'], check=True)
# O Git LFS pode deixar somente um ponteiro de texto no clone. Baixe os bytes reais.
from urllib.request import urlretrieve
import hashlib
DATASET = RAIZ / 'audio/datasets/dataset_audio_emocoes_1180.zip'
if not DATASET.exists() or DATASET.stat().st_size < 1000:
    commit = subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True).strip()
    urlretrieve(f'https://media.githubusercontent.com/media/Val-Faria/TechChallengeFase4/{commit}/audio/datasets/dataset_audio_emocoes_1180.zip', DATASET)
with DATASET.open('rb') as stream:
    digest = hashlib.file_digest(stream, 'sha256').hexdigest()
assert digest == 'd37a28e118f648be967c37224d27f9a54cb1bdd76446be30033235d5f74f83e6', 'Dataset diferente da versão auditada'
print('Dataset:', DATASET.stat().st_size, 'bytes; SHA-256 conferido')


## Extração e avaliação

A primeira execução baixa o modelo Whisper tiny (aproximadamente 72 MB). As características ficam em cache, com hashes para permitir retomada. O texto dos metadados é usado somente para auditoria e agrupamento; a transcrição vem do Whisper. O relatório contém separadamente validação, teste e holdout.


In [ ]:
subprocess.run([sys.executable, 'scripts/audio_reproduce.py', '--device', 'auto'], check=True)


In [ ]:
import json
relatorio = json.loads((RAIZ / 'evidence/audio_reproducao.json').read_text(encoding='utf-8'))
for particao in ['validation', 'test', 'blind_holdout']:
    m = relatorio[particao]
    print(particao, 'n=', m['n'], 'acurácia=', round(m['accuracy'], 4), 'macro-F1=', round(m['macro_f1'], 4))
print('Sobreposição de frases, originais e hashes:', relatorio['leakage'])


## Persistir antes de desconectar

O Drive é montado na conta de quem executa o notebook. O pacote abaixo inclui modelo, CSVs, relatório de métricas e manifesto de partições. Não contém a chave da OpenAI. Faça a autorização do Drive somente se quiser essa cópia.


In [ ]:
SALVAR_NO_DRIVE = True
if EM_COLAB and SALVAR_NO_DRIVE:
    from google.colab import drive
    import shutil
    drive.mount('/content/drive')
    DESTINO = Path('/content/drive/MyDrive/TechChallenge4/entrega_final/audio')
    DESTINO.mkdir(parents=True, exist_ok=True)
    shutil.copytree(RAIZ / 'artifacts/audio', DESTINO / 'artifacts', dirs_exist_ok=True)
    shutil.copytree(RAIZ / 'audio/evidencias', DESTINO / 'evidencias', dirs_exist_ok=True)
    shutil.copy2(RAIZ / 'evidence/audio_reproducao.json', DESTINO / 'audio_reproducao.json')
    print('Cópia persistente salva:', DESTINO)
else:
    print('Resultados locais persistidos em artifacts/audio, audio/evidencias e evidence.')


## Encerrar sessão de GPU

Depois de conferir a cópia salva, execute a célula final. Ela desconecta e exclui somente o ambiente temporário do Colab; os arquivos copiados ao Drive permanecem. Localmente, não há sessão Colab a encerrar.


In [ ]:
ENCERRAR_COLAB_APOS_SALVAR = False  # marque True somente depois de conferir os arquivos no Drive
if EM_COLAB and ENCERRAR_COLAB_APOS_SALVAR:
    from google.colab import runtime
    runtime.unassign()
